Q-Learning MDP (CORE LOGIC)

In [1]:
# ================================
# 📌 1. IMPORTS
# ================================
import pandas as pd
import random
from collections import defaultdict

# ================================
# 📌 2. LOAD DATA
# ================================
ratings = pd.read_csv("/content/ratings.csv")
movies = pd.read_csv("/content/movies_metadata.csv", low_memory=False)

# Convert movie IDs properly
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')

# ================================
# 📌 3. PREPROCESSING
# ================================

# Sort ratings by user and time
ratings = ratings.sort_values(by=["userId", "timestamp"])

# Reduce dataset size (IMPORTANT 🚨)
top_movies = ratings["movieId"].value_counts().head(500).index
ratings = ratings[ratings["movieId"].isin(top_movies)]

states = ratings["movieId"].unique()

# ================================
# 📌 4. BUILD TRANSITIONS
# ================================
def build_transitions(ratings_df):
    transitions = []

    for user_id, user_ratings in ratings_df.groupby('userId'):
        movie_seq = user_ratings["movieId"].tolist()
        rating_seq = user_ratings["rating"].tolist()

        for i in range(len(movie_seq) - 1):
            state = movie_seq[i]
            next_state = movie_seq[i + 1]
            reward = rating_seq[i + 1]

            transitions.append((state, next_state, reward))

    return transitions

transitions = build_transitions(ratings)

# ================================
# 📌 5. REWARD FUNCTION
# ================================
def reward_function(rating):
    if rating >= 4:
        return 1
    elif rating >= 3:
        return 0.5
    else:
        return -1

# ================================
# 📌 6. MDP RECOMMENDER CLASS
# ================================
class MDPRecommender:
    def __init__(self, states, alpha=0.1, gamma=0.9, epsilon=0.2):
        self.states = states
        self.q = defaultdict(float)  # Sparse Q-table
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

        # Build transition map for valid actions
        self.transition_map = self.build_transition_map(transitions)

    def build_transition_map(self, transitions):
        transition_map = defaultdict(set)
        for s, next_s, _ in transitions:
            transition_map[s].add(next_s)
        return transition_map

    def get_possible_actions(self, state):
        return list(self.transition_map[state]) if state in self.transition_map else []

    def choose_action(self, state):
        actions = self.get_possible_actions(state)
        if not actions:
            return random.choice(self.states)

        if random.random() < self.epsilon:
            return random.choice(actions)

        return max(actions, key=lambda a: self.q[(state, a)])

    def train(self, transitions, episodes=5000):
        for _ in range(episodes):
            state, next_state, rating = random.choice(transitions)

            reward = reward_function(rating)
            actions_next = self.get_possible_actions(next_state)

            if actions_next:
                best_next = max(actions_next, key=lambda a: self.q[(next_state, a)])
            else:
                best_next = next_state

            # Q-learning update
            self.q[(state, next_state)] += self.alpha * (
                reward +
                self.gamma * self.q[(next_state, best_next)] -
                self.q[(state, next_state)]
            )

            # Decay epsilon (exploration ↓ over time)
            self.epsilon *= 0.999

    def recommend(self, state, top_n=5):
        actions = self.get_possible_actions(state)

        if not actions:
            return []

        scored = [(a, self.q[(state, a)]) for a in actions]
        scored.sort(key=lambda x: x[1], reverse=True)

        return [a for a, _ in scored[:top_n]]

# ================================
# 📌 7. TRAIN MODEL
# ================================
agent = MDPRecommender(states)
agent.train(transitions)

# ================================
# 📌 8. MOVIE ID → TITLE MAPPING
# ================================
movie_titles = movies[['id', 'title']].dropna()
movie_dict = dict(zip(movie_titles['id'], movie_titles['title']))

# ================================
# 📌 9. TEST RECOMMENDATION
# ================================
test_movie = states[0]  # pick any movie

recommendations = agent.recommend(test_movie)

print("🎬 Recommended Movies:\n")
for movie_id in recommendations:
    print(movie_dict.get(movie_id, f"Movie ID: {movie_id}"))

🎬 Recommended Movies:

Longitude
Movie ID: 1704
Movie ID: 3897
Men in Black II
The Goddess
